# Practice: trees, forests and boosting on house prices
*AI from scratch, chapter 13* · lesson: https://ihor.sh/ai/13-trees-and-boosting.html

The models that win most tabular problems at work, on 20,640 California districts (median house value from
income, age, rooms, location…): a linear baseline, a decision tree and its depth, a random forest, gradient
boosting with early stopping, and what the model looks at.
Fill in each `...` under `# YOUR CODE HERE`, then run the check under it (✓ = right, ✗ = not yet).
On Kaggle, turn on *Internet* in the notebook settings for the download.

In [ ]:
# Helpers: run this cell first. check() marks an answer ✓ or ✗; stuck? hint(n) gives the next hint for
# exercise n, answer(n) shows a solution.
import base64 as _b64
_HINTS = {1: ['Compute MAE of predicting the training mean ytr.mean() for all test examples: np.full(len(yte), ytr.mean()).', 'Fit LinearRegression().fit(Xtr, ytr) and compute its test error with mae(linear_model).', 'mae_mean = mean_absolute_error(yte, np.full(len(yte), ytr.mean()))', 'mae_linear = mae(LinearRegression().fit(Xtr, ytr))'], 2: ['Initialize tree_train = [] and tree_test = [].', 'For each dpt in depths, fit DecisionTreeRegressor(max_depth=dpt, random_state=0) on (Xtr, ytr).', 'Record train error using mean_absolute_error(ytr, t.predict(Xtr)) and test error using mae(t).', 'Find best_depth = depths[int(np.argmin(tree_test))].', 'Loop over dpt in depths and fit t = DecisionTreeRegressor(max_depth=dpt, random_state=0).fit(Xtr, ytr).', 'tree_train.append(mean_absolute_error(ytr, t.predict(Xtr))) and tree_test.append(mae(t)).', 'best_depth = depths[int(np.argmin(tree_test))].'], 3: ['Fit RandomForestRegressor(n_estimators=100, n_jobs=-1, random_state=0) on (Xtr, ytr).', 'Compute test MAE using mae(forest).', 'forest = RandomForestRegressor(n_estimators=100, n_jobs=-1, random_state=0).fit(Xtr, ytr)', 'mae_forest = mae(forest)'], 4: ['Fit HistGradientBoostingRegressor(max_iter=1000, learning_rate=0.1, early_stopping=True, random_state=0) on (Xtr, ytr).', 'Compute test MAE: mae_boost = mae(boost).', 'Record the number of trees grown before early stopping: n_trees = boost.n_iter_.', 'boost = HistGradientBoostingRegressor(max_iter=1000, learning_rate=0.1, early_stopping=True, random_state=0).fit(Xtr, ytr)', 'mae_boost, n_trees = mae(boost), boost.n_iter_'], 5: ['Run permutation_importance(boost, Xte, yte, scoring="neg_mean_absolute_error", n_repeats=3, random_state=0).', 'Extract r.importances_mean and pair with column names X.columns.', 'Sort descending by importance and store as a dict {feature: mean_error_increase}.', 'r = permutation_importance(boost, Xte, yte, scoring="neg_mean_absolute_error", n_repeats=3, random_state=0)', 'Pair feature names and importances with zip(X.columns, r.importances_mean) and sort descending.', 'importance = dict(sorted(zip(X.columns, r.importances_mean), key=lambda kv: -kv[1]))']}
_ANSWERS = {1: 'bWFlX21lYW4gPSBtZWFuX2Fic29sdXRlX2Vycm9yKHl0ZSwgbnAuZnVsbChsZW4oeXRlKSwgeXRyLm1lYW4oKSkpCm1hZV9saW5lYXIgPSBtYWUoTGluZWFyUmVncmVzc2lvbigpLmZpdChYdHIsIHl0cikp', 2: 'dHJlZV90cmFpbiwgdHJlZV90ZXN0ID0gW10sIFtdCmZvciBkcHQgaW4gZGVwdGhzOgogICAgdCA9IERlY2lzaW9uVHJlZVJlZ3Jlc3NvcihtYXhfZGVwdGg9ZHB0LCByYW5kb21fc3RhdGU9MCkuZml0KFh0ciwgeXRyKQogICAgdHJlZV90cmFpbi5hcHBlbmQobWVhbl9hYnNvbHV0ZV9lcnJvcih5dHIsIHQucHJlZGljdChYdHIpKSk7IHRyZWVfdGVzdC5hcHBlbmQobWFlKHQpKQpiZXN0X2RlcHRoID0gZGVwdGhzW2ludChucC5hcmdtaW4odHJlZV90ZXN0KSldCnBsdC5wbG90KGRlcHRocywgdHJlZV90cmFpbiwgIm8tIiwgbGFiZWw9InRyYWluIik7IHBsdC5wbG90KGRlcHRocywgdHJlZV90ZXN0LCAiby0iLCBsYWJlbD0idGVzdCIpOyBwbHQubGVnZW5kKCk7IHBsdC54bGFiZWwoIm1heF9kZXB0aCIpOyBwbHQuc2hvdygp', 3: 'Zm9yZXN0ID0gUmFuZG9tRm9yZXN0UmVncmVzc29yKG5fZXN0aW1hdG9ycz0xMDAsIG5fam9icz0tMSwgcmFuZG9tX3N0YXRlPTApLmZpdChYdHIsIHl0cikKbWFlX2ZvcmVzdCA9IG1hZShmb3Jlc3Qp', 4: 'Ym9vc3QgPSBIaXN0R3JhZGllbnRCb29zdGluZ1JlZ3Jlc3NvcihtYXhfaXRlcj0xMDAwLCBsZWFybmluZ19yYXRlPTAuMSwgZWFybHlfc3RvcHBpbmc9VHJ1ZSwgcmFuZG9tX3N0YXRlPTApLmZpdChYdHIsIHl0cikKbWFlX2Jvb3N0LCBuX3RyZWVzID0gbWFlKGJvb3N0KSwgYm9vc3Qubl9pdGVyXw==', 5: 'ciA9IHBlcm11dGF0aW9uX2ltcG9ydGFuY2UoYm9vc3QsIFh0ZSwgeXRlLCBzY29yaW5nPSJuZWdfbWVhbl9hYnNvbHV0ZV9lcnJvciIsIG5fcmVwZWF0cz0zLCByYW5kb21fc3RhdGU9MCkKaW1wb3J0YW5jZSA9IGRpY3Qoc29ydGVkKHppcChYLmNvbHVtbnMsIHIuaW1wb3J0YW5jZXNfbWVhbiksIGtleT1sYW1iZGEga3Y6IC1rdlsxXSkp'}
_given = {}

def check(n, test, msg=""):
    try:
        ok = bool(test())
    except NameError as e:
        return print("✗ %s: run the exercise cell above first." % e)
    except Exception as e:
        if "NoneType" in str(e) or "ellipsis" in str(e):
            return print("✗ Not answered yet: replace the ... in exercise %d and run that cell. Stuck? hint(%d)" % (n, n))
        ok, msg = False, msg + " (the check stopped with %s: %s)" % (type(e).__name__, e)
    print("✓ Right." if ok else "✗ Not yet. " + msg + "\n  Stuck? hint(%d) for a hint, answer(%d) for a solution." % (n, n))

def hint(n):
    hints, i = _HINTS.get(n, []), _given.get(n, 0)
    if not hints:
        return print("No hints for this one: answer(%d) shows a solution." % n)
    print("Hint %d of %d: %s" % (min(i, len(hints) - 1) + 1, len(hints), hints[min(i, len(hints) - 1)]))
    _given[n] = i + 1

def answer(n):
    print(_b64.b64decode(_ANSWERS[n]).decode())
    print("\n(Put this in place of the ..., run it, then close it and try writing it yourself.)")


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error

data = fetch_california_housing(as_frame=True)
X, y = data.data, data.target                       # target: median house value in $100,000s
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, random_state=0)
mae = lambda m: mean_absolute_error(yte, m.predict(Xte))
X.head()

## 1. A baseline first
Always start with something simple to beat:
- Set `mae_mean`: the error of predicting the training mean (`ytr.mean()`) for everyone using `mean_absolute_error`.
- Set `mae_linear`: the error of `LinearRegression` evaluated with `mae(...)`.
(Errors in $100k: 0.5 means $50,000 off on average.)

In [ ]:
mae_mean = mae_linear = None
# YOUR CODE HERE (stuck? hint(1), then answer(1))
...
print("Baseline (mean):", mae_mean, "Linear regression:", mae_linear)

In [ ]:
check(1, lambda: mae_linear is not None and mae_mean is not None and mae_linear < mae_mean * 0.65, 
      "a straight line should beat guessing the average by at least 35% in mean absolute error")

## 2. A tree's depth is its complexity knob
For depths 2, 4, 6, 8, 12, 16, 24, fit `DecisionTreeRegressor(max_depth=d, random_state=0)` and record train and
test MAE. Set `best_depth` (lowest test error). Deep trees memorize: train error → 0, test error rises.

In [ ]:
depths = [2, 4, 6, 8, 12, 16, 24]
tree_train = tree_test = best_depth = None
# YOUR CODE HERE (stuck? hint(2), then answer(2))
...
print("best depth:", best_depth)

In [ ]:
check(2, lambda: tree_train is not None and tree_train[-1] < 0.05 and best_depth is not None and 6 <= best_depth <= 12, 
      "a 24-deep tree memorizes (train MAE under 0.05), and the best test depth should be in the sweet spot, 6 to 12")

## 3. A forest of trees
A random forest averages many deep trees, each grown on a bootstrap sample with random feature subsets: their
mistakes partly cancel. Fit `RandomForestRegressor(n_estimators=100, n_jobs=-1, random_state=0)`; set `mae_forest`.

In [ ]:
mae_forest = None
# YOUR CODE HERE (stuck? hint(3), then answer(3))
...
print("Random forest MAE:", mae_forest)

In [ ]:
check(3, lambda: mae_forest is not None and tree_test is not None and mae_forest < min(tree_test) * 0.85, 
      "the random forest should beat the best single tree by at least 15%")

## 4. Gradient boosting with early stopping
`HistGradientBoostingRegressor` is scikit-learn's fast boosting (the same family as XGBoost and LightGBM). Give it
up to 1000 trees, `learning_rate=0.1` and `early_stopping=True`: it holds out 10% of the training data and stops
when that stops improving. Set `mae_boost` and `n_trees` (`.n_iter_`).

In [ ]:
mae_boost = n_trees = None
# YOUR CODE HERE (stuck? hint(4), then answer(4))
...
print("Boosting MAE:", mae_boost, "after", n_trees, "trees")

In [ ]:
check(4, lambda: mae_boost is not None and mae_linear is not None and mae_boost < mae_linear * 0.65 and n_trees is not None and n_trees < 1000, 
      "boosting should beat the straight line by over 35% and stop early (fewer than 1,000 trees)")

## 5. What does the model look at?
**Permutation importance**: shuffle one column of the test set and see how much the error grows. Set
`importance`: a dict `{feature: mean increase in error}` using `permutation_importance(boost, Xte, yte, scoring="neg_mean_absolute_error", n_repeats=3, random_state=0)`.
Read each feature's mean error increase from `importances_mean` and visualize with `plt.barh`.

In [ ]:
from sklearn.inspection import permutation_importance
importance = None
# YOUR CODE HERE (stuck? hint(5), then answer(5))
...
if importance:
    plt.barh(list(importance)[::-1], list(importance.values())[::-1])
    plt.xlabel("error increase when shuffled")
    plt.show()
print(importance)

In [ ]:
check(5, lambda: importance is not None and list(importance)[0] in ("MedInc", "Latitude", "Longitude"), 
      f"The most important feature ({list(importance)[0] if importance else None}) should be MedInc, Latitude, or Longitude.")

## Next
Back to the path: https://ihor.sh/ai/ · Stage 3 milestone: take these tools to a real Kaggle competition.